# 07 — RQ3 step 2: do learned network embeddings improve the source ranker?

`06` showed that, with age-matched candidates, a hand-built ranker clearly beats popularity (MRR 0.345 vs 0.252). This notebook asks whether a **learned node2vec embedding** adds affinity signal that the hand-built features miss.

**Approach.**
- **Snapshots:** for each **half-year** snapshot (built only from remix links that existed before the snapshot date), we learn node2vec embeddings of tracks.
- **Taste vector:** a remixer's taste vector is the mean embedding of the sources they used before time `T`.
- **Affinity feature:** the cosine similarity between the taste vector and a candidate's embedding is added to the eight features from `06`.
- **Comparability:** both vectors come from the **same** snapshot, so the cosine is comparable across events.

**Fair comparison.**
- **Same candidates:** evaluation uses exactly the candidate sets frozen by `06` (`rq3_eval_sets.pkl`), for the recent and age-matched regimes.
- **Same setup:** the metrics and random tie-breaking are the same as in `06`.
- **Same training:** both rankers are trained on age-matched negatives, as in `06`.

In [1]:
import numpy as np, pandas as pd, time, pickle, networkx as nx
from bisect import bisect_left
from collections import defaultdict
from node2vec import Node2Vec
from sklearn.ensemble import HistGradientBoostingClassifier
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
DAY=86400
nc=pd.read_csv("data/processed/nodes_clean.csv"); loc=pd.read_csv("data/processed/edges.csv"); loc=loc[loc.edge_type=="local"].copy()
CUT=pd.Timestamp("2022-01-01",tz="UTC").timestamp()
id2t=dict(zip(nc.upload_id,nc.date_unix.astype("int64"))); id2auth=dict(zip(nc.upload_id,nc.user_name)); id2ns=dict(zip(nc.upload_id,nc.n_sources))
def toks(s): return set() if pd.isna(s) else {x.strip() for x in str(s).replace(";",",").split(",") if x.strip()}
id2tags=dict(zip(nc.upload_id,nc.usertags.apply(toks)))
srt=np.argsort(nc.date_unix.astype("int64").values); allids_s=nc.upload_id.values[srt]; allt_s=nc.date_unix.astype("int64").values[srt]
parent_kids={p:np.sort(s.child_date_unix.values) for p,s in loc.groupby("parent_id")}
auth_times={a:np.sort(g.values) for a,g in nc.groupby("user_name").date_unix}
ca=loc.child_id.map(id2auth); given={a:np.sort(loc.child_date_unix[ca==a].values) for a in ca.unique()}
prof={}
for a,g in nc.groupby("user_name"):
    gg=g.sort_values("date_unix"); prof[a]=(gg.date_unix.astype("int64").values,[id2tags[u] for u in gg.upload_id])
rem_srcauth=defaultdict(lambda: defaultdict(list))
for c,p,cd in zip(loc.child_id,loc.parent_id,loc.child_date_unix): rem_srcauth[id2auth[c]][id2auth[p]].append(cd)
for a in rem_srcauth:
    for b in rem_srcauth[a]: rem_srcauth[a][b]=np.sort(rem_srcauth[a][b])
rem_src=defaultdict(list)
for c,p,cd in zip(loc.child_id,loc.parent_id,loc.child_date_unix): rem_src[id2auth[c]].append((cd,p))
rem_ct={a:np.array([x[0] for x in sorted(v)]) for a,v in rem_src.items()}
rem_id={a:[x[1] for x in sorted(v)] for a,v in rem_src.items()}
child_srcs=loc.groupby("child_id").parent_id.apply(set).to_dict()
def asof(a,T): return 0 if a is None else int(bisect_left(a,T))
def prof_tags(a,T):
    p=prof.get(a)
    if not p: return set()
    ts,tg=p; s=set()
    for i in range(bisect_left(ts,T)): s|=tg[i]
    return s
def used_auth(rem,sa,T):
    d=rem_srcauth.get(rem); return 0 if (not d or sa not in d) else int(bisect_left(d[sa],T)>0)
def feats(rem,sid,T,pt):
    sset=id2tags.get(sid,set()); jac=(len(pt&sset)/len(pt|sset)) if (pt or sset) else 0.0
    return [asof(parent_kids.get(sid),T),(T-id2t[sid])/DAY,id2ns.get(sid,0),int(id2auth.get(sid)==rem),
            asof(auth_times.get(rem),T),asof(given.get(rem),T),jac,used_auth(rem,id2auth.get(sid),T)]
def cap(T): return bisect_left(allt_s,T)
AGE_WIN=90*DAY
def negs_agematched(T,true,excl,k,r):
    ts=id2t[true]; lo=bisect_left(allt_s,ts-AGE_WIN); hi=min(bisect_left(allt_s,ts+AGE_WIN),cap(T))
    out=set(); g=0
    while len(out)<k and hi-lo>1 and g<k*20:
        s=allids_s[r.integers(lo,hi)]; g+=1
        if s not in excl: out.add(s)
    return list(out)
ev=loc[["child_id","parent_id","child_date_unix"]].copy(); ev["rem"]=ev.child_id.map(id2auth)
train_ev=ev[ev.child_date_unix<CUT].sample(12000,random_state=0)
print("events",len(loc))

events 59186


## 1. Load the frozen candidate sets from `06`

In [2]:
with open("data/processed/rq3_eval_sets.pkl","rb") as fh: EVAL=pickle.load(fh)
EVAL={k:EVAL[k] for k in ["recent","agematched"]}
for k,v in EVAL.items(): print(f"{k}: {len(v)} events")

recent: 5672 events
agematched: 5672 events


## 2. As-of node2vec embeddings (half-year snapshots)

One embedding is learned per half-year period used by any training or test event, 43 in total (32 dimensions, `seed=0`). A track only has an embedding if it had at least one remix link before the snapshot date.

In [3]:
def period(T):
    ts=pd.Timestamp(T,unit="s",tz="UTC"); return ts.year*2+int(ts.month>6)
def period_start(p):
    return int(pd.Timestamp(f"{p//2}-{7 if p%2 else 1:02d}-01",tz="UTC").timestamp())
need=sorted({period(T) for T in train_ev.child_date_unix}|{period(x[3]) for v in EVAL.values() for x in v})
t0=time.time(); emb={}
for p in need:
    sub=loc[loc.child_date_unix<period_start(p)]
    if len(sub)<50: emb[p]=None; continue
    G=nx.from_pandas_edgelist(sub,"parent_id","child_id",create_using=nx.DiGraph())
    mm=Node2Vec(G,dimensions=32,walk_length=15,num_walks=5,workers=4,quiet=True,seed=0).fit(window=5,min_count=1,epochs=2,workers=4,seed=0)
    emb[p]={int(k):mm.wv[k] for k in mm.wv.index_to_key}
print(f"node2vec: {sum(v is not None for v in emb.values())} half-year snapshots in {time.time()-t0:.0f}s")
def E_at(T): return emb.get(period(T))

node2vec: 43 half-year snapshots in 242s


## 3. Taste vectors and embedding coverage

Coverage is reported explicitly, because a missing embedding gives a cosine of 0.

**Result.**
- Only **62.9 %** of true sources have an embedding at the time of the remix. The others had no earlier remix links, so they are not in the graph.
- 66.6 % of age-matched negatives have an embedding (74.5 % of recent negatives).
- 99 % of remixers have a taste vector.

In [4]:
def centroid(rem,T,E):
    cts=rem_ct.get(rem)
    if cts is None or E is None: return None
    k=bisect_left(cts,T)
    vs=[E[s] for s in rem_id[rem][:k] if s in E]
    return np.mean(vs,axis=0) if vs else None
def cos(a,b):
    if a is None or b is None: return 0.0
    n=np.linalg.norm(a)*np.linalg.norm(b); return float(np.dot(a,b)/n) if n>0 else 0.0
for k,rows in EVAL.items():
    t_cov=c_cov=r_cov=0; n_c=0
    for c,rem,ps,T,ng in rows:
        E=E_at(T) or {}
        t_cov+=ps in E; c_cov+=sum(s in E for s in ng); n_c+=len(ng); r_cov+=centroid(rem,T,E) is not None
    n=len(rows)
    print(f"{k:11s} true source embedded {t_cov/n:.1%} | negatives embedded {c_cov/n_c:.1%} | remixer has taste vector {r_cov/n:.1%}")

recent      true source embedded 62.9% | negatives embedded 74.5% | remixer has taste vector 99.0%
agematched  true source embedded 62.9% | negatives embedded 66.6% | remixer has taste vector 99.0%


## 4. Train base vs base + embedding

Both rankers are trained on age-matched negatives. The base ranker uses the eight features from `06`, so it reproduces `06`'s age-matched model. The second ranker adds two features: the taste cosine, and a flag for whether both embeddings exist.

In [5]:
def emb_feats(rem,sid,T,cen,E):
    se=E.get(sid) if E else None
    return [cos(cen,se),int(cen is not None and se is not None)]
def build(with_emb,seed=12):
    r=np.random.default_rng(seed); X=[]; y=[]
    for c,rem,ps,T in zip(train_ev.child_id,train_ev.rem,train_ev.parent_id,train_ev.child_date_unix):
        pt=prof_tags(rem,T); E=E_at(T); cen=centroid(rem,T,E) if with_emb else None
        for lab,sid in [(1,ps)]+[(0,s) for s in negs_agematched(T,ps,child_srcs[c],10,r)]:
            f=feats(rem,sid,T,pt)
            if with_emb: f+=emb_feats(rem,sid,T,cen,E)
            X.append(f); y.append(lab)
    return HistGradientBoostingClassifier(max_depth=4,learning_rate=0.05,max_iter=300,random_state=0).fit(np.array(X),np.array(y))
t0=time.time(); m_base=build(False); m_emb=build(True); print(f"trained both in {time.time()-t0:.0f}s")

trained both in 13s


## 5. Evaluation on the frozen candidate sets

For each regime, the table shows MRR (with 95 % CI) and Hits@K for popularity, the embedding score alone, the base ranker, and base + embedding. A paired bootstrap gives the difference between base + embedding and base.

In [6]:
def rank_of_true(s,r):
    s=np.asarray(s,float); return 1+int((s[1:]>s[0]).sum())+int(r.integers(0,int((s[1:]==s[0]).sum())+1))
def score_all(rows,r):
    rk={k:[] for k in ["popularity","embedding-only","base (=06 model)","base+embedding"]}
    for c,rem,ps,T,ng in rows:
        cands=[ps]+ng; pt=prof_tags(rem,T); E=E_at(T); cen=centroid(rem,T,E)
        Xb=np.array([feats(rem,s,T,pt) for s in cands])
        Xe=np.hstack([Xb,np.array([emb_feats(rem,s,T,cen,E) for s in cands])])
        sc={"popularity":Xb[:,0],"embedding-only":Xe[:,-2],
            "base (=06 model)":m_base.predict_proba(Xb)[:,1],"base+embedding":m_emb.predict_proba(Xe)[:,1]}
        for m,s in sc.items(): rk[m].append(rank_of_true(s,r))
    return {m:np.array(v) for m,v in rk.items()}
def summarize(ranks,r,B=1000):
    n=len(next(iter(ranks.values()))); idx=r.integers(0,n,(B,n)); out={}
    for m,v in ranks.items():
        rr=1/v; bt=rr[idx].mean(1)
        out[m]=dict(MRR=rr.mean(),MRR_lo=np.percentile(bt,2.5),MRR_hi=np.percentile(bt,97.5),H1=(v<=1).mean(),H5=(v<=5).mean(),H10=(v<=10).mean())
    return pd.DataFrame(out).T
TABLES={}
for k,rows in EVAL.items():
    R=score_all(rows,np.random.default_rng(30)); TABLES[k]=summarize(R,np.random.default_rng(31))
    d=1/R["base+embedding"]-1/R["base (=06 model)"]; idx=np.random.default_rng(32).integers(0,len(d),(1000,len(d))); bt=d[idx].mean(1)
    print(f"\n=== negatives: {k} ===\n{TABLES[k].round(3).to_string()}")
    print(f"paired MRR diff (base+embedding - base): {d.mean():+.3f}  95% CI [{np.percentile(bt,2.5):+.3f}, {np.percentile(bt,97.5):+.3f}]")
pd.concat(TABLES,names=["negatives","method"]).round(4).to_csv("results_rq3_embedding.csv")


=== negatives: recent ===
                    MRR  MRR_lo  MRR_hi     H1     H5    H10
popularity        0.309   0.300   0.319  0.202  0.391  0.520
embedding-only    0.089   0.086   0.094  0.021  0.098  0.181
base (=06 model)  0.372   0.362   0.381  0.237  0.500  0.658
base+embedding    0.368   0.359   0.378  0.234  0.498  0.655
paired MRR diff (base+embedding - base): -0.003  95% CI [-0.006, -0.001]

=== negatives: agematched ===
                    MRR  MRR_lo  MRR_hi     H1     H5    H10
popularity        0.250   0.243   0.259  0.128  0.353  0.514
embedding-only    0.091   0.086   0.095  0.022  0.098  0.192
base (=06 model)  0.345   0.336   0.354  0.195  0.500  0.675
base+embedding    0.344   0.336   0.353  0.193  0.504  0.674
paired MRR diff (base+embedding - base): -0.001  95% CI [-0.003, +0.003]


## 6. Figure (age-matched regime)

In [7]:
t=TABLES["agematched"]; fig,ax=plt.subplots(figsize=(7,4)); x=np.arange(len(t)); w=.38
ax.bar(x-w/2,t.MRR,w,yerr=[t.MRR-t.MRR_lo,t.MRR_hi-t.MRR],capsize=3,label="MRR (95% CI)",color="#4c72b0")
ax.bar(x+w/2,t.H10,w,label="Hits@10",color="#55a868")
ax.set_xticks(x); ax.set_xticklabels(t.index,rotation=18,ha="right",fontsize=8)
ax.set_title("RQ3 embedding ranker, age-matched negatives"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("fig_07_rq3_embedding.png",dpi=130); print("saved fig_07_rq3_embedding.png")

saved fig_07_rq3_embedding.png


## 7. Conclusion

  **Learned embeddings add nothing, and in the recent regime they slightly hurt.**

  | Regime | Base ranker MRR | Base + embedding MRR | Paired difference (95 % CI) | Embedding alone |
  |---|---|---|---|---|
  | Age-matched | 0.345 | 0.344 | −0.001 [−0.003, +0.003] | 0.091 (random guess ≈ 0.09) |
  | Recent | 0.372 | 0.368 | −0.003 [−0.006, −0.001] | 0.089 (random guess ≈ 0.09) |

  Hits@10 is essentially unchanged (age-matched: 0.675 vs 0.674). On its own, the embedding score is at chance level. The small negative effect in the recent regime is not stable across node2vec refits (an earlier run gave −0.002 [−0.004, +0.001]), so it should not be relied on.

**Why.**
- **Coverage:** node2vec can only embed tracks that already have remix links. 37 % of true sources have no embedding at the time of the remix, and those are exactly the not-yet-proven sources that personalisation would need to find.
- **Redundancy:** where embeddings do exist, their information is already captured by the hand-built features (source popularity, age, and past remixer–author ties).
- **Sparse data:** only 162 remixers are active in the test window.

**RQ3 overall.** The findings are consistent across all four notebooks:
- Network position adds no signal to remixability prediction (`05` / `05b`).
- Remix sources are recommendable, and a simple learned ranker beats popularity by about 0.09 MRR under age-matched evaluation (`06`).
- Learned network embeddings add nothing on top of that ranker (`07`).

On ccMixter, both *whether* a track is remixed and *which* source a remixer picks are explained by simple track-, author- and relationship-level signals, not by learned collaborative structure. This supports the decision not to build a full GNN.